[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/AI-in-healthcare-book/blob/main/notebooks/chapter_08/notebook_8_2_ner.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Clinical Named Entity Recognition (NER)

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Implement rule-based NER** using regex and medical dictionaries
2. **Build CRF-based NER models** using sequence features
3. **Understand BioBERT/ClinicalBERT** for clinical entity extraction
4. **Evaluate NER systems** using precision, recall, and F1-score
5. **Handle entity boundaries** and nested entities in clinical text

## Clinical Context

**Clinical Scenario**: Automated extraction of clinical entities from discharge summaries:
- **Diseases**: "heart failure", "pneumonia", "diabetes"
- **Medications**: "furosemide 40mg", "lisinopril", "metformin"
- **Tests**: "BNP", "chest X-ray", "echocardiography"
- **Findings**: "elevated", "normal", "reduced ejection fraction"

**Applications**:
- Automated clinical coding (ICD-10, CPT)
- Adverse event detection from clinical notes
- Clinical trial cohort identification
- Quality metrics extraction

**Challenge**: Medical entities are complex:
- Multi-word: "acute decompensated heart failure"
- Nested: "[left ventricular [ejection fraction]]"
- Discontinuous: "X-ray of the chest"
- Ambiguous: "discharge" (verb vs. noun)

---

## Setup

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import defaultdict, Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_recall_fscore_support
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("✓ Libraries imported successfully!")

## 1. Generate Annotated Clinical Text

We'll create synthetic clinical sentences with entity annotations in BIO (Begin-Inside-Outside) format:
- **B-DISEASE**: Beginning of disease entity
- **I-DISEASE**: Inside disease entity
- **B-MEDICATION**: Beginning of medication entity
- **I-MEDICATION**: Inside medication entity
- **B-TEST**: Beginning of test/procedure entity
- **I-TEST**: Inside test entity
- **O**: Outside any entity

In [ ]:
# Synthetic clinical sentences with BIO annotations
annotated_sentences = [
    # Sentence 1
    {
        'tokens': ['Patient', 'diagnosed', 'with', 'heart', 'failure', 'and', 'started', 'on', 'furosemide', '.'],
        'labels': ['O', 'O', 'O', 'B-DISEASE', 'I-DISEASE', 'O', 'O', 'O', 'B-MEDICATION', 'O']
    },
    # Sentence 2
    {
        'tokens': ['Chest', 'X-ray', 'showed', 'pulmonary', 'edema', '.'],
        'labels': ['B-TEST', 'I-TEST', 'O', 'B-DISEASE', 'I-DISEASE', 'O']
    },
    # Sentence 3
    {
        'tokens': ['BNP', 'elevated', 'at', '850', 'pg/mL', '.'],
        'labels': ['B-TEST', 'O', 'O', 'O', 'O', 'O']
    },
    # Sentence 4
    {
        'tokens': ['Patient', 'has', 'diabetes', 'mellitus', 'type', '2', ',', 'taking', 'metformin', '1000mg', '.'],
        'labels': ['O', 'O', 'B-DISEASE', 'I-DISEASE', 'I-DISEASE', 'I-DISEASE', 'O', 'O', 'B-MEDICATION', 'I-MEDICATION', 'O']
    },
    # Sentence 5
    {
        'tokens': ['Echocardiography', 'revealed', 'reduced', 'ejection', 'fraction', 'of', '35%', '.'],
        'labels': ['B-TEST', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
    },
    # Sentence 6
    {
        'tokens': ['No', 'evidence', 'of', 'pneumonia', 'on', 'CT', 'scan', '.'],
        'labels': ['O', 'O', 'O', 'B-DISEASE', 'O', 'B-TEST', 'I-TEST', 'O']
    },
    # Sentence 7
    {
        'tokens': ['Started', 'lisinopril', '10mg', 'for', 'hypertension', '.'],
        'labels': ['O', 'B-MEDICATION', 'I-MEDICATION', 'O', 'B-DISEASE', 'O']
    },
    # Sentence 8
    {
        'tokens': ['MRI', 'of', 'brain', 'negative', 'for', 'stroke', '.'],
        'labels': ['B-TEST', 'O', 'O', 'O', 'O', 'B-DISEASE', 'O']
    },
    # Sentence 9
    {
        'tokens': ['Acute', 'myocardial', 'infarction', 'confirmed', 'by', 'troponin', 'levels', '.'],
        'labels': ['B-DISEASE', 'I-DISEASE', 'I-DISEASE', 'O', 'O', 'B-TEST', 'I-TEST', 'O']
    },
    # Sentence 10
    {
        'tokens': ['Patient', 'on', 'aspirin', '81mg', 'and', 'atorvastatin', '40mg', 'daily', '.'],
        'labels': ['O', 'O', 'B-MEDICATION', 'I-MEDICATION', 'O', 'B-MEDICATION', 'I-MEDICATION', 'O', 'O']
    },
    # Sentence 11
    {
        'tokens': ['Blood', 'glucose', 'test', 'showed', 'hyperglycemia', '.'],
        'labels': ['B-TEST', 'I-TEST', 'I-TEST', 'O', 'B-DISEASE', 'O']
    },
    # Sentence 12
    {
        'tokens': ['Chronic', 'obstructive', 'pulmonary', 'disease', 'managed', 'with', 'albuterol', 'inhaler', '.'],
        'labels': ['B-DISEASE', 'I-DISEASE', 'I-DISEASE', 'I-DISEASE', 'O', 'O', 'B-MEDICATION', 'I-MEDICATION', 'O']
    },
    # Sentence 13
    {
        'tokens': ['Ultrasound', 'of', 'abdomen', 'normal', '.'],
        'labels': ['B-TEST', 'O', 'O', 'O', 'O']
    },
    # Sentence 14
    {
        'tokens': ['History', 'of', 'coronary', 'artery', 'disease', ',', 'taking', 'clopidogrel', '.'],
        'labels': ['O', 'O', 'B-DISEASE', 'I-DISEASE', 'I-DISEASE', 'O', 'O', 'B-MEDICATION', 'O']
    },
    # Sentence 15
    {
        'tokens': ['EKG', 'showed', 'atrial', 'fibrillation', '.'],
        'labels': ['B-TEST', 'O', 'B-DISEASE', 'I-DISEASE', 'O']
    },
]

print(f"Generated {len(annotated_sentences)} annotated sentences")
print(f"\nExample sentence with BIO tags:")
print("="*60)
example = annotated_sentences[0]
for token, label in zip(example['tokens'], example['labels']):
    print(f"{token:15s} -> {label}")

# Count entity types
all_labels = [label for sent in annotated_sentences for label in sent['labels']]
label_counts = Counter(all_labels)
print(f"\nLabel distribution:")
for label, count in sorted(label_counts.items()):
    print(f"  {label:15s}: {count:3d}")

## 2. Rule-Based NER

**Approach**: Use medical dictionaries and regex patterns to identify entities.

**Advantages**:
- No training data required
- Transparent and interpretable
- Fast inference

**Disadvantages**:
- Requires manual curation of dictionaries
- Poor generalization to unseen terms
- Difficult to handle ambiguity

In [ ]:
class RuleBasedNER:
    """
    Dictionary and pattern-based NER system.
    """

    def __init__(self):
        # Medical entity dictionaries (simplified for demonstration)
        self.disease_terms = [
            'heart failure', 'diabetes mellitus', 'hypertension',
            'pneumonia', 'stroke', 'myocardial infarction',
            'pulmonary edema', 'diabetes', 'coronary artery disease',
            'chronic obstructive pulmonary disease', 'COPD',
            'atrial fibrillation', 'hyperglycemia', 'CHF'
        ]

        self.medication_terms = [
            'furosemide', 'lisinopril', 'metformin', 'aspirin',
            'atorvastatin', 'clopidogrel', 'albuterol',
            'warfarin', 'insulin', 'losartan'
        ]

        self.test_terms = [
            'chest x-ray', 'BNP', 'echocardiography', 'CT scan',
            'MRI', 'troponin', 'EKG', 'ECG', 'ultrasound',
            'blood glucose test'
        ]

        # Sort by length (longest first) for greedy matching
        self.disease_terms.sort(key=len, reverse=True)
        self.medication_terms.sort(key=len, reverse=True)
        self.test_terms.sort(key=len, reverse=True)

    def predict(self, tokens):
        """
        Predict BIO tags for tokens using dictionary matching.

        Args:
            tokens: List of tokens

        Returns:
            List of BIO tags
        """
        labels = ['O'] * len(tokens)
        text = ' '.join(tokens).lower()

        # Find all entity matches
        matches = []

        # Match diseases
        for term in self.disease_terms:
            for match in re.finditer(r'\b' + re.escape(term.lower()) + r'\b', text):
                matches.append({
                    'start': match.start(),
                    'end': match.end(),
                    'type': 'DISEASE',
                    'text': term
                })

        # Match medications
        for term in self.medication_terms:
            # Match medication with optional dosage (e.g., "furosemide 40mg")
            pattern = r'\b' + re.escape(term.lower()) + r'(?:\s+\d+\s*mg)?\b'
            for match in re.finditer(pattern, text):
                matches.append({
                    'start': match.start(),
                    'end': match.end(),
                    'type': 'MEDICATION',
                    'text': match.group()
                })

        # Match tests
        for term in self.test_terms:
            pattern = r'\b' + re.escape(term.lower()) + r'\b'
            for match in re.finditer(pattern, text):
                matches.append({
                    'start': match.start(),
                    'end': match.end(),
                    'type': 'TEST',
                    'text': term
                })

        # Resolve overlaps (keep longest match)
        matches.sort(key=lambda x: (x['start'], -(x['end'] - x['start'])))
        non_overlapping = []
        last_end = -1
        for match in matches:
            if match['start'] >= last_end:
                non_overlapping.append(match)
                last_end = match['end']

        # Convert character positions to token positions
        token_positions = []
        current_pos = 0
        for i, token in enumerate(tokens):
            start = text.find(token.lower(), current_pos)
            if start == -1:
                # Fallback if exact match not found
                start = current_pos
            end = start + len(token)
            token_positions.append((start, end))
            current_pos = end

        # Assign BIO tags
        for match in non_overlapping:
            match_tokens = []
            for i, (tok_start, tok_end) in enumerate(token_positions):
                # Check if token overlaps with match
                if tok_start < match['end'] and tok_end > match['start']:
                    match_tokens.append(i)

            # Assign B- and I- tags
            if match_tokens:
                labels[match_tokens[0]] = f"B-{match['type']}"
                for i in match_tokens[1:]:
                    labels[i] = f"I-{match['type']}"

        return labels

# Test rule-based NER
rule_ner = RuleBasedNER()

print("Rule-Based NER Examples:\n")
print("="*60)

for i, example in enumerate(annotated_sentences[:3]):
    tokens = example['tokens']
    true_labels = example['labels']
    pred_labels = rule_ner.predict(tokens)

    print(f"\nSentence {i+1}: {' '.join(tokens)}")
    print(f"\n{'Token':15s} {'True':15s} {'Predicted':15s} {'Match':5s}")
    print("-"*55)

    for token, true_label, pred_label in zip(tokens, true_labels, pred_labels):
        match = '✓' if true_label == pred_label else '✗'
        print(f"{token:15s} {true_label:15s} {pred_label:15s} {match:5s}")

print("\n" + "="*60)

## 3. Feature Engineering for CRF

**Conditional Random Fields (CRF)**: Sequence labeling algorithm that considers context.

**Key Features**:
1. **Word features**: Capitalization, suffixes, prefixes
2. **Context features**: Previous/next words
3. **Orthographic features**: Contains digits, punctuation
4. **Dictionary features**: Matches medical term

In [ ]:
class CRFFeatureExtractor:
    """
    Extract features for CRF-based NER.
    """

    def __init__(self, rule_ner=None):
        self.rule_ner = rule_ner

    def word_features(self, word):
        """
        Extract word-level features.
        """
        features = {
            'word.lower': word.lower(),
            'word.isupper': word.isupper(),
            'word.istitle': word.istitle(),
            'word.isdigit': word.isdigit(),
            'word.length': len(word),
            'word.prefix-2': word[:2].lower() if len(word) >= 2 else '',
            'word.prefix-3': word[:3].lower() if len(word) >= 3 else '',
            'word.suffix-2': word[-2:].lower() if len(word) >= 2 else '',
            'word.suffix-3': word[-3:].lower() if len(word) >= 3 else '',
            'word.has-digit': any(c.isdigit() for c in word),
            'word.has-hyphen': '-' in word,
        }
        return features

    def sentence_features(self, tokens, index):
        """
        Extract features for a token in context.

        Args:
            tokens: List of tokens in sentence
            index: Index of target token

        Returns:
            Dictionary of features
        """
        word = tokens[index]
        features = self.word_features(word)

        # Context features: previous word
        if index > 0:
            prev_word = tokens[index - 1]
            features.update({
                '-1:word.lower': prev_word.lower(),
                '-1:word.istitle': prev_word.istitle(),
                '-1:word.isupper': prev_word.isupper(),
            })
        else:
            features['BOS'] = True  # Beginning of sentence

        # Context features: next word
        if index < len(tokens) - 1:
            next_word = tokens[index + 1]
            features.update({
                '+1:word.lower': next_word.lower(),
                '+1:word.istitle': next_word.istitle(),
                '+1:word.isupper': next_word.isupper(),
            })
        else:
            features['EOS'] = True  # End of sentence

        # Dictionary features (if rule-based NER available)
        if self.rule_ner:
            pred_labels = self.rule_ner.predict(tokens)
            features['dict-label'] = pred_labels[index]

        return features

    def extract_features(self, sentences):
        """
        Extract features for all sentences.

        Args:
            sentences: List of {'tokens': [...], 'labels': [...]}

        Returns:
            Tuple of (X_features, y_labels)
        """
        X = []
        y = []

        for sent in sentences:
            tokens = sent['tokens']
            labels = sent['labels']

            # Extract features for each token
            sent_features = []
            for i in range(len(tokens)):
                features = self.sentence_features(tokens, i)
                sent_features.append(features)

            X.append(sent_features)
            y.append(labels)

        return X, y

# Test feature extraction
feature_extractor = CRFFeatureExtractor(rule_ner=rule_ner)

example_sent = annotated_sentences[0]
example_features = feature_extractor.sentence_features(example_sent['tokens'], 3)

print("Feature Extraction Example:")
print("="*60)
print(f"Sentence: {' '.join(example_sent['tokens'])}")
print(f"\nFeatures for token '{example_sent['tokens'][3]}' (index 3):")
print()
for feature_name, feature_value in sorted(example_features.items()):
    print(f"  {feature_name:20s}: {feature_value}")

## 4. Simple CRF Implementation (Simulation)

**Note**: Full CRF training requires specialized libraries like `sklearn-crfsuite` or `python-crfsuite`.
For this educational notebook, we'll simulate CRF behavior using feature-based classification.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

class SimplifiedCRF:
    """
    Simplified sequence tagger using Random Forest on CRF-style features.

    Note: This is a simplified version for educational purposes.
    True CRF models (using Viterbi decoding) would perform better.
    """

    def __init__(self, feature_extractor):
        self.feature_extractor = feature_extractor
        self.model = RandomForestClassifier(n_estimators=100, random_state=42)
        self.label_encoder = LabelEncoder()

    def features_to_vector(self, features):
        """
        Convert feature dictionary to numerical vector.
        """
        # Simple feature hashing for demonstration
        vector = []

        # Extract specific features (simplified)
        vector.append(1 if features.get('word.isupper', False) else 0)
        vector.append(1 if features.get('word.istitle', False) else 0)
        vector.append(1 if features.get('word.isdigit', False) else 0)
        vector.append(features.get('word.length', 0))
        vector.append(1 if features.get('word.has-digit', False) else 0)
        vector.append(1 if features.get('word.has-hyphen', False) else 0)
        vector.append(1 if features.get('BOS', False) else 0)
        vector.append(1 if features.get('EOS', False) else 0)

        # Dictionary label encoding
        dict_label = features.get('dict-label', 'O')
        dict_encoding = {'O': 0, 'B-DISEASE': 1, 'I-DISEASE': 2,
                        'B-MEDICATION': 3, 'I-MEDICATION': 4,
                        'B-TEST': 5, 'I-TEST': 6}
        vector.append(dict_encoding.get(dict_label, 0))

        return vector

    def fit(self, X_features, y_labels):
        """
        Train the model.

        Args:
            X_features: List of sentence features (list of list of dicts)
            y_labels: List of sentence labels (list of lists)
        """
        # Flatten sentences into individual tokens
        X_flat = []
        y_flat = []

        for sent_features, sent_labels in zip(X_features, y_labels):
            for features, label in zip(sent_features, sent_labels):
                X_flat.append(self.features_to_vector(features))
                y_flat.append(label)

        X_flat = np.array(X_flat)
        y_flat = np.array(y_flat)

        # Encode labels
        y_encoded = self.label_encoder.fit_transform(y_flat)

        # Train model
        self.model.fit(X_flat, y_encoded)

        print(f"Model trained on {len(X_flat)} tokens")
        print(f"Label classes: {list(self.label_encoder.classes_)}")

    def predict(self, X_features):
        """
        Predict labels for new sentences.

        Args:
            X_features: List of sentence features

        Returns:
            List of predicted labels for each sentence
        """
        predictions = []

        for sent_features in X_features:
            X_sent = np.array([self.features_to_vector(f) for f in sent_features])
            y_pred_encoded = self.model.predict(X_sent)
            y_pred = self.label_encoder.inverse_transform(y_pred_encoded)
            predictions.append(list(y_pred))

        return predictions

# Split data into train/test
train_sentences = annotated_sentences[:12]
test_sentences = annotated_sentences[12:]

print(f"Training set: {len(train_sentences)} sentences")
print(f"Test set: {len(test_sentences)} sentences")

# Extract features
X_train, y_train = feature_extractor.extract_features(train_sentences)
X_test, y_test = feature_extractor.extract_features(test_sentences)

# Train simplified CRF
crf_model = SimplifiedCRF(feature_extractor)
crf_model.fit(X_train, y_train)

# Predict on test set
y_pred = crf_model.predict(X_test)

print("\nTest Set Predictions:\n")
print("="*60)

for i, (sent, true_labels, pred_labels) in enumerate(zip(test_sentences, y_test, y_pred)):
    tokens = sent['tokens']
    print(f"\nSentence {i+1}: {' '.join(tokens)}")
    print(f"\n{'Token':15s} {'True':15s} {'Predicted':15s} {'Match':5s}")
    print("-"*55)

    for token, true_label, pred_label in zip(tokens, true_labels, pred_labels):
        match = '✓' if true_label == pred_label else '✗'
        print(f"{token:15s} {true_label:15s} {pred_label:15s} {match:5s}")

## 5. Evaluation Metrics

**Token-level metrics**: Treat each token independently
- Precision = TP / (TP + FP)
- Recall = TP / (TP + FN)
- F1 = 2 × (Precision × Recall) / (Precision + Recall)

**Entity-level metrics**: Require exact entity boundary match
- Stricter evaluation: "heart failure" predicted as "heart" + "failure" counts as 2 errors

In [ ]:
def evaluate_ner(y_true_all, y_pred_all, model_name="Model"):
    """
    Evaluate NER predictions at token level.

    Args:
        y_true_all: List of true label sequences
        y_pred_all: List of predicted label sequences
        model_name: Name for display
    """
    # Flatten all labels
    y_true_flat = [label for sent in y_true_all for label in sent]
    y_pred_flat = [label for sent in y_pred_all for label in sent]

    # Get unique labels (excluding 'O')
    all_labels = sorted(set(y_true_flat + y_pred_flat))
    entity_labels = [l for l in all_labels if l != 'O']

    print(f"\n{'='*70}")
    print(f"{model_name} Evaluation (Token-level)")
    print(f"{'='*70}\n")

    # Overall metrics
    report = classification_report(y_true_flat, y_pred_flat,
                                   labels=entity_labels,
                                   zero_division=0)
    print(report)

    # Confusion matrix for major entity types
    from sklearn.metrics import confusion_matrix

    # Simplify to entity type (B-/I- -> type)
    def get_entity_type(label):
        if label == 'O':
            return 'O'
        return label.split('-')[1]  # B-DISEASE -> DISEASE

    y_true_types = [get_entity_type(l) for l in y_true_flat]
    y_pred_types = [get_entity_type(l) for l in y_pred_flat]

    entity_types = ['O', 'DISEASE', 'MEDICATION', 'TEST']
    cm = confusion_matrix(y_true_types, y_pred_types, labels=entity_types)

    # Plot confusion matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=entity_types,
                yticklabels=entity_types)
    plt.xlabel('Predicted Entity Type')
    plt.ylabel('True Entity Type')
    plt.title(f'{model_name} - Confusion Matrix (Entity Types)')
    plt.tight_layout()
    plt.show()

    return report

# Evaluate rule-based NER
y_rule_pred = [rule_ner.predict(sent['tokens']) for sent in test_sentences]
evaluate_ner(y_test, y_rule_pred, model_name="Rule-Based NER")

# Evaluate CRF-based NER
evaluate_ner(y_test, y_pred, model_name="CRF-Based NER")

## 6. BioBERT / ClinicalBERT (Conceptual Overview)

**Transformer-based NER**: State-of-the-art approach using pre-trained language models.

### Architecture

```
Input: "Patient has heart failure"
       ↓
[Tokenization]
       ↓
[BioBERT Encoder] (12-24 transformer layers)
       ↓
[Token Embeddings] (768-dim vectors per token)
       ↓
[Classification Head] (Linear layer)
       ↓
Output: [O, O, B-DISEASE, I-DISEASE]
```

### Key Models

| Model | Training Data | Advantages |
|-------|---------------|------------|
| **BioBERT** | PubMed + PMC (biomedical literature) | Best for scientific/research text |
| **ClinicalBERT** | MIMIC-III clinical notes | Best for clinical narratives |
| **BlueBERT** | PubMed + MIMIC-III | Hybrid approach |
| **PubMedBERT** | PubMed abstracts only | Strong biomedical vocabulary |

### Implementation (Conceptual)

```python
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

# Load pre-trained ClinicalBERT
model_name = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list)  # B-DISEASE, I-DISEASE, etc.
)

# Fine-tune on labeled clinical NER data
# ... training loop ...

# Inference
text = "Patient has heart failure"
inputs = tokenizer(text, return_tensors="pt")
outputs = model(**inputs)
predictions = torch.argmax(outputs.logits, dim=2)
```

### Performance Comparison

Typical F1-scores on i2b2/n2c2 clinical NER benchmarks:

| Approach | F1-Score |
|----------|----------|
| Rule-based | 0.65-0.75 |
| CRF | 0.75-0.85 |
| BiLSTM-CRF | 0.80-0.88 |
| **BioBERT/ClinicalBERT** | **0.88-0.92** |

### Trade-offs

**Advantages**:
- State-of-the-art performance
- Captures complex context
- Transfer learning from large corpora

**Disadvantages**:
- Requires GPU for training/inference
- Large model size (~400MB)
- Needs labeled training data
- Less interpretable than rule-based

## 7. Error Analysis

In [ ]:
def analyze_errors(y_true_all, y_pred_all, sentences, model_name="Model"):
    """
    Analyze common error patterns.
    """
    print(f"\n{'='*70}")
    print(f"{model_name} - Error Analysis")
    print(f"{'='*70}\n")

    errors = []

    for sent, y_true, y_pred in zip(sentences, y_true_all, y_pred_all):
        tokens = sent['tokens']

        for i, (token, true_label, pred_label) in enumerate(zip(tokens, y_true, y_pred)):
            if true_label != pred_label:
                # Get context
                context_start = max(0, i-2)
                context_end = min(len(tokens), i+3)
                context = ' '.join(tokens[context_start:context_end])

                errors.append({
                    'token': token,
                    'context': context,
                    'true': true_label,
                    'pred': pred_label,
                    'error_type': f"{true_label} → {pred_label}"
                })

    if not errors:
        print("No errors found!")
        return

    # Count error types
    error_type_counts = Counter([e['error_type'] for e in errors])

    print("Most common error types:\n")
    for error_type, count in error_type_counts.most_common(10):
        print(f"  {error_type:40s}: {count:2d}")

    print(f"\n\nExample errors (first 5):\n")
    for i, error in enumerate(errors[:5]):
        print(f"{i+1}. Token: '{error['token']}'")
        print(f"   Context: {error['context']}")
        print(f"   True: {error['true']:15s} | Predicted: {error['pred']}")
        print()

# Analyze errors for both models
analyze_errors(y_test, y_rule_pred, test_sentences, model_name="Rule-Based NER")
analyze_errors(y_test, y_pred, test_sentences, model_name="CRF-Based NER")

## 8. Real-World Considerations

### Clinical Deployment Challenges

| Challenge | Impact | Solution |
|-----------|--------|----------|
| **Entity boundary errors** | "acute heart failure" tagged as [acute] [heart failure] | Use entity-level evaluation, retrain on more examples |
| **Nested entities** | "[left ventricular [systolic dysfunction]]" | Specialized nested NER models or hierarchical tagging |
| **Abbreviation ambiguity** | "MS" = Multiple Sclerosis vs. Mitral Stenosis | Context-aware models (BERT), abbreviation expansion |
| **Multi-word entities** | "Type 2 diabetes mellitus" split incorrectly | Use CRF/BERT that considers context |
| **Domain shift** | Model trained on cardiology notes fails on oncology | Multi-domain training or domain adaptation |

### Production Best Practices

1. **Data Annotation**:
   - Use double annotation (2 annotators per document)
   - Inter-annotator agreement (Cohen's κ > 0.80)
   - Clear annotation guidelines for edge cases
   - Regular calibration sessions

2. **Model Selection**:
   - **High-resource**: BioBERT/ClinicalBERT (F1 ~0.90)
   - **Medium-resource**: CRF with rich features (F1 ~0.82)
   - **Low-resource**: Rule-based + dictionary (F1 ~0.70)
   - **Hybrid**: Ensemble rule-based + ML

3. **Evaluation**:
   - Token-level AND entity-level metrics
   - Per-entity-type performance analysis
   - Clinical expert review of errors
   - Test on held-out hospitals/departments

4. **Monitoring**:
   - Track prediction confidence scores
   - Flag low-confidence predictions for review
   - Regular retraining on new data
   - Detect distribution drift

### Common Pitfalls

❌ **Wrong**: Ignoring entity boundaries
```python
# BAD: Only evaluates token-level accuracy
accuracy = (y_true == y_pred).mean()
```

✅ **Correct**: Entity-level evaluation
```python
# GOOD: Extracts full entities and compares
true_entities = extract_entities(tokens, y_true)  # [(start, end, type)]
pred_entities = extract_entities(tokens, y_pred)
f1 = compute_entity_f1(true_entities, pred_entities)
```

---

❌ **Wrong**: Training only on one hospital's notes
```python
# BAD: Overfits to institution-specific templates
train_data = load_notes(hospital="Hospital A")
model.fit(train_data)
```

✅ **Correct**: Multi-site training and evaluation
```python
# GOOD: Trains on diverse data, tests on unseen hospital
train_data = load_notes(hospitals=["A", "B", "C"])
test_data = load_notes(hospitals=["D"])  # Held-out hospital
model.fit(train_data)
evaluate(model, test_data)
```

## Key Takeaways

### What We Learned

1. **Rule-Based NER**
   - Fast and interpretable
   - Requires manual dictionary curation
   - Baseline F1 ~0.65-0.75
   - Good for high-precision applications

2. **CRF-Based NER**
   - Uses sequence context (previous/next words)
   - Feature engineering is critical
   - F1 ~0.75-0.85 with good features
   - Balance of performance and interpretability

3. **Transformer-Based NER (BioBERT/ClinicalBERT)**
   - State-of-the-art performance (F1 ~0.88-0.92)
   - Requires GPU and labeled data
   - Best for high-stakes applications
   - Less interpretable

4. **Evaluation**
   - Token-level metrics can be misleading
   - Entity-level F1 is gold standard
   - Error analysis reveals systematic failures
   - Per-entity-type performance varies widely

### Connections to Book Chapters

- **Chapter 8.1**: Text preprocessing prepares data for NER
- **Chapter 8.3**: BERT classification extends to sequence labeling
- **Chapter 8.4**: LLM prompting can extract entities zero-shot
- **Journey 7**: Clinical coding relies on accurate NER

### Clinical Validation Requirements

Before deploying in clinical settings:

1. **Entity-level F1** > 0.85 for each critical entity type
2. **Clinical expert review** of 100+ error cases
3. **Multi-site validation** (test on ≥ 2 unseen hospitals)
4. **Inter-annotator agreement** κ > 0.80 on test set
5. **Confidence calibration** (flag uncertain predictions)

---

## Exercises

1. **Expand Entity Types**: Add "SYMPTOM" and "ANATOMY" entity types. Create annotated examples and retrain the models.

2. **Improve Features**: Add more CRF features like word shape ("Aa", "AA", "aa"), POS tags, or word embeddings. Measure impact on F1.

3. **Handle Nested Entities**: Implement detection of nested entities like "[left ventricular [ejection fraction]]". How would you modify the BIO tagging scheme?

4. **Entity Linking**: After extracting entities, link them to standard vocabularies (UMLS, SNOMED-CT, RxNorm). How would you handle ambiguity?

5. **Active Learning**: Implement an active learning loop that selects the most uncertain predictions for human annotation.

6. **Error Analysis**: Categorize errors into: boundary errors, type confusion, missed entities, spurious entities. Which is most common?

7. **Domain Adaptation**: Train on cardiology notes, test on oncology notes. Measure performance drop. How can you improve cross-domain generalization?

8. **Real-Time NER**: Implement a fast inference pipeline that processes clinical notes in <100ms per document. What optimizations are needed?

---

*This notebook is part of "AI in Healthcare" (Volume 1)*  
*Full implementation available in the complete book companion code*